# AeroPulse Anomaly — 09: Rolling-Origin Evaluation

Replace the single chronological split with expanding-window folds, and find
out whether the hazard model works in every season or only in the one that
happened to land in the test slice. Implements plan section 4.

## The evaluation weakness this fixes

Every result in notebooks 03–08 comes from one 70/15/15 chronological cut,
whose test window is **entirely monsoon** — the cleanest season of the Indian
year. Plan section 4 identifies this as a major weakness, and it is the
single biggest caveat on everything reported so far.

Rolling-origin validation trains on an expanding prefix and validates on the
window that follows, so each season gets its turn as the held-out period:

```
fold 1  train |=====|            valid |==|
fold 2  train |========|         valid |==|
fold 3  train |===========|      valid |==|
fold 4  train |==============|   valid |==|
                             ^gap^
```

Each fold keeps a **48-hour embargo** between train and validation. Without
it, a 48h rolling feature computed at the start of validation would overlap
the training window — a subtle leak that a plain chronological cut also has,
at exactly one point in the series.

### The limitation this notebook cannot fix

The dataset spans 19 months, February 2025 to September 2026. That contains
**one** full winter. A winter fold can therefore be *validated* but never
*replicated*: there is no second winter to check whether fold performance was
a property of the season or of that particular winter's meteorology.

Plan section 28 asks for 3–5 years. This notebook is the concrete argument
for why that matters, and it is the reason no per-season number here should
be treated as a stable estimate.

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


In [2]:
# ============================================================================
# LOAD
# ============================================================================
import lightgbm as lgb

manifest = json.loads((CFG["feature_dir"] / "dataset_manifest.json").read_text())
df = pd.read_parquet(FEATURE_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

H = CFG["horizon_hours"]
LABEL = f"target_very_poor_within_{H}h"
df = df[df[LABEL].notna()].sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
FEATURES = [c for c in manifest["feature_candidates"]
            if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

span_days = (df["timestamp_utc"].max() - df["timestamp_utc"].min()).days
print(f"{len(df):,} rows / {span_days} days / {span_days/365.25:.1f} years")
print(f"seasons present: {df['season'].value_counts().to_dict()}")

1,705,103 rows / 567 days / 1.6 years
seasons present: {'monsoon': 672066, 'pre_monsoon': 562242, 'winter': 280954, 'post_monsoon': 189841}


In [3]:
# ============================================================================
# FOLD CONSTRUCTION
# ============================================================================
folds = tk.rolling_origin_folds(df["timestamp_utc"], n_folds=5,
                                min_train_days=180, valid_days=75, embargo_hours=48)
print(f"{len(folds)} folds\n")
fold_info = pd.DataFrame([tk.describe_fold(df, f, LABEL) for f in folds])
print(fold_info.to_string(index=False))

pv = fold_info["valid_prevalence"]
print(f"\nhazard prevalence across folds: {pv.min():.4%} -> {pv.max():.4%} "
      f"({pv.max()/max(pv.min(), 1e-9):.0f}x swing)")

5 folds

 fold  train_rows  valid_rows valid_start  valid_end           valid_seasons  train_prevalence  valid_prevalence
    1      550181      234066  2025-08-19 2025-11-02 [monsoon, post_monsoon]          0.086177          0.085664
    2      784046      221881  2025-11-02 2026-01-16  [post_monsoon, winter]          0.084884          0.311613
    3     1012201      212908  2026-01-16 2026-04-01   [pre_monsoon, winter]          0.135105          0.141817
    4     1218912      228362  2026-04-01 2026-06-15  [monsoon, pre_monsoon]          0.136641          0.074272
    5     1447096      219574  2026-06-15 2026-08-29               [monsoon]          0.126954          0.031119

hazard prevalence across folds: 3.1119% -> 31.1613% (10x swing)


### Result — the number that invalidates a single-split evaluation

Hazard prevalence runs from **3.1% (fold 5, monsoon) to 31.2% (fold 2,
post-monsoon + winter)** — a 10× swing from the same stations and the same
label definition.

The monsoon test slice used throughout notebooks 03–08 sits at the **bottom**
of that range. Every headline number in those notebooks was measured in the
least polluted quarter of the year.

Three consequences:

1. **PR-AUC is not comparable across folds.** Precision is bounded above by
   prevalence, so a winter fold posts a higher PR-AUC partly for arithmetic
   reasons. Lift over prevalence is the comparable quantity.
2. **A fixed probability threshold cannot be right in both seasons.** Section 3
   below measures exactly how wrong it gets.
3. **The plan's acceptance criteria are prevalence-dependent.** "Precision
   ≥ 0.25" is a much harder ask at 3% prevalence than at 31%.

## 1. Run the folds

In [4]:
# ============================================================================
# ROLLING-ORIGIN TRAINING
# ============================================================================
params = dict(objective="binary", metric="average_precision", learning_rate=0.05,
              num_leaves=127, min_data_in_leaf=200, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=os.cpu_count() or 4, verbosity=-1, seed=42)
BUDGET = CFG["false_alert_budget"]

results, preds = [], []
for f in folds:
    tr = df[(df["timestamp_utc"] >= f["train_start"]) & (df["timestamp_utc"] < f["train_end"])]
    va = df[(df["timestamp_utc"] >= f["valid_start"]) & (df["timestamp_utc"] < f["valid_end"])]
    if tr[LABEL].sum() < 100 or va[LABEL].sum() < 20:
        print(f"fold {f['fold']}: too few positives, skipped")
        continue

    # Threshold is chosen on the tail of TRAIN, never on the fold's own
    # validation window — otherwise every fold reports an oracle threshold.
    cut = tr["timestamp_utc"].quantile(0.85)
    fit, cal = tr[tr["timestamp_utc"] < cut], tr[tr["timestamp_utc"] >= cut]
    d = lgb.Dataset(fit[FEATURES], label=fit[LABEL], free_raw_data=False)
    dv = lgb.Dataset(cal[FEATURES], label=cal[LABEL], reference=d)
    t0 = time.time()
    mdl = lgb.train(params, d, num_boost_round=500, valid_sets=[dv],
                    callbacks=[lgb.early_stopping(40, verbose=False)])
    p_cal = mdl.predict(cal[FEATURES], num_iteration=mdl.best_iteration)
    p_va = mdl.predict(va[FEATURES], num_iteration=mdl.best_iteration)
    thr, _ = tk.choose_threshold(cal[LABEL], p_cal, budget_false_alert_rate=BUDGET)
    m = tk.classification_metrics(va[LABEL], p_va, thr)

    # Persistence, evaluated on the same fold, as the reference to beat.
    thr_p, _ = tk.choose_threshold(cal[LABEL], cal["pm25_lag_1h"],
                                   budget_false_alert_rate=BUDGET)
    m_p = tk.classification_metrics(va[LABEL], va["pm25_lag_1h"], thr_p)

    seasons = "+".join(sorted(va["season"].unique()))
    results.append({
        "fold": f["fold"], "valid_start": str(f["valid_start"].date()),
        "seasons": seasons, "valid_rows": len(va), "prevalence": m["prevalence"],
        "pr_auc": m["pr_auc"], "roc_auc": m["roc_auc"], "precision": m["precision"],
        "recall": m["recall"], "f1": m["f1"],
        "lift": m["pr_auc"] / m["prevalence"] if m["prevalence"] else np.nan,
        "persistence_pr_auc": m_p["pr_auc"],
        "persistence_lift": m_p["pr_auc"] / m_p["prevalence"] if m_p["prevalence"] else np.nan,
    })
    preds.append(va[["location_id", "timestamp_utc", "season", LABEL]].assign(
        prob=p_va, fold=f["fold"]))
    print(f"fold {f['fold']} {seasons:<24} prev {m['prevalence']:.3%}  "
          f"PR-AUC {m['pr_auc']:.4f} (lift {results[-1]['lift']:.1f}x)  "
          f"R {m['recall']:.3f}  ({time.time()-t0:.0f}s)")

folds_df = pd.DataFrame(results)

fold 1 monsoon+post_monsoon     prev 8.566%  PR-AUC 0.5604 (lift 6.5x)  R 0.744  (11s)


fold 2 post_monsoon+winter      prev 31.161%  PR-AUC 0.8230 (lift 2.6x)  R 0.762  (14s)


fold 3 pre_monsoon+winter       prev 14.182%  PR-AUC 0.6332 (lift 4.5x)  R 0.359  (27s)


fold 4 monsoon+pre_monsoon      prev 7.427%  PR-AUC 0.4435 (lift 6.0x)  R 0.293  (44s)


fold 5 monsoon                  prev 3.112%  PR-AUC 0.4138 (lift 13.3x)  R 0.388  (33s)


In [5]:
# ============================================================================
# FOLD SUMMARY
# ============================================================================
print(folds_df[["fold", "seasons", "prevalence", "pr_auc", "lift",
                "persistence_lift", "precision", "recall", "roc_auc"]].round(4).to_string(index=False))
print()
for col in ("pr_auc", "lift", "recall", "roc_auc"):
    s = folds_df[col]
    print(f"{col:<10} mean {s.mean():.4f}  std {s.std():.4f}  "
          f"min {s.min():.4f}  max {s.max():.4f}")
beats = int((folds_df["lift"] > folds_df["persistence_lift"]).sum())
print(f"\nfolds where the model beats persistence on lift: {beats}/{len(folds_df)}")

 fold              seasons  prevalence  pr_auc    lift  persistence_lift  precision  recall  roc_auc
    1 monsoon+post_monsoon      0.0857  0.5604  6.5414            5.1215     0.3154  0.7443   0.8769
    2  post_monsoon+winter      0.3116  0.8230  2.6412            2.2667     0.7212  0.7622   0.8945
    3   pre_monsoon+winter      0.1418  0.6332  4.4646            3.1198     0.7646  0.3593   0.8720
    4  monsoon+pre_monsoon      0.0743  0.4435  5.9707            3.7315     0.5993  0.2933   0.8429
    5              monsoon      0.0311  0.4138 13.2957            6.1265     0.4945  0.3878   0.8798

pr_auc     mean 0.5747  std 0.1646  min 0.4138  max 0.8230
lift       mean 6.5827  std 4.0455  min 2.6412  max 13.2957
recall     mean 0.5094  std 0.2253  min 0.2933  max 0.7622
roc_auc    mean 0.8732  std 0.0189  min 0.8429  max 0.8945

folds where the model beats persistence on lift: 5/5


### Result — the ranking is stable; everything prevalence-dependent is not

This is the most reassuring table in the whole pipeline, and the split
between the two kinds of metric is the point.

| Metric | mean | std | min | max |
|---|---:|---:|---:|---:|
| **ROC-AUC** | ~0.873 | **0.019** | 0.843 | 0.895 |
| PR-AUC | ~0.575 | 0.165 | 0.414 | 0.823 |
| lift | ~6.6 | 4.05 | 2.6 | 13.3 |
| recall | ~0.509 | 0.225 | 0.293 | 0.762 |

**ROC-AUC varies by less than 0.06 across the entire year.** Because ROC-AUC
is prevalence-invariant, this says the model's *ability to rank* hazardous
station-hours above clean ones is essentially season-independent. Plan
section 34 asks that "no major season should collapse" — by the one metric
that can fairly answer that question, none does.

PR-AUC, lift and recall swing widely, and all three are prevalence-coupled.
Their spread is a property of the seasons, not evidence of an unstable model.

**The model beats persistence on lift in 5 of 5 folds** — including monsoon,
where notebook 03's residual detector *lost* to persistence. That reversal is
the clearest single confirmation that the objective change worked.

Two folds also clear plan section 34's joint gates outright:

| Fold | Seasons | Precision | Recall | Both gates? |
|---|---|---:|---:|---|
| 1 | monsoon + post-monsoon | 0.32 | 0.74 | **yes** |
| 2 | post-monsoon + winter | 0.72 | 0.76 | **yes** |
| 3 | pre-monsoon + winter | 0.76 | 0.36 | recall short |
| 4 | monsoon + pre-monsoon | 0.60 | 0.29 | recall short |
| 5 | monsoon | 0.49 | 0.39 | recall short |

The folds that clear both are the higher-prevalence ones — which is also
where a public-health alert matters most.

## 2. Per-season performance

In [6]:
# ============================================================================
# POOLED OUT-OF-FOLD PREDICTIONS BY SEASON
# ============================================================================
oof = pd.concat(preds, ignore_index=True)
print(f"{len(oof):,} out-of-fold predictions")

rows = []
for season, g in oof.groupby("season"):
    if g[LABEL].sum() < 20:
        print(f"  {season}: only {int(g[LABEL].sum())} positives, skipped")
        continue
    m = tk.classification_metrics(g[LABEL], g["prob"], g["prob"].quantile(0.95))
    rows.append({"season": season, "rows": len(g), "positives": int(g[LABEL].sum()),
                 "prevalence": round(m["prevalence"], 5), "pr_auc": round(m["pr_auc"], 4),
                 "roc_auc": round(m["roc_auc"], 4),
                 "lift": round(m["pr_auc"] / m["prevalence"], 1) if m["prevalence"] else np.nan,
                 "recall_at_top5pct": round(m["recall"], 4),
                 "precision_at_top5pct": round(m["precision"], 4)})
season_table = pd.DataFrame(rows).sort_values("prevalence", ascending=False)
print()
print(season_table.to_string(index=False))

1,116,791 out-of-fold predictions



      season   rows  positives  prevalence  pr_auc  roc_auc  lift  recall_at_top5pct  precision_at_top5pct
      winter 247036      62550     0.25320  0.7654   0.8934   3.0             0.1790                0.9067
post_monsoon 189841      40892     0.21540  0.7465   0.8841   3.5             0.2177                0.9377
 pre_monsoon 282100      26178     0.09280  0.4993   0.8472   5.4             0.3349                0.6216
     monsoon 397814      13560     0.03409  0.3554   0.8446  10.4             0.4805                0.3275


### Result — per-season, at a fixed alert rate

Each season is scored at *its own* 95th percentile, holding the alert **rate**
constant so that differences reflect the model rather than the threshold.

| Season | Prevalence | ROC-AUC | Lift | Recall @top5% | Precision @top5% |
|---|---:|---:|---:|---:|---:|
| winter | 25.3% | 0.893 | 3.0 | 0.18 | **0.91** |
| post-monsoon | 21.5% | 0.884 | 3.5 | 0.22 | **0.94** |
| pre-monsoon | 9.3% | 0.847 | 5.4 | 0.33 | 0.62 |
| monsoon | 3.4% | 0.845 | 10.4 | 0.48 | 0.33 |

Read the two ends together. In **winter, precision at a 5% alert rate is
~0.91** — nine in ten alerts are real — but recall is only 0.18, because 25%
of station-hours are hazardous and a 5% budget cannot cover them. In
**monsoon the position inverts**: recall 0.48 at precision 0.33.

That is a budget constraint, not a model failure, and it has a direct product
consequence: **the alert budget should scale with the season.** A fixed 5% of
station-hours is far too tight for winter and roughly right for monsoon.
Precision this high in winter means the budget could be widened substantially
before users start seeing noise.

## 3. Does the threshold transfer between seasons?

The direct test of the concern above: pick a threshold on one season, apply
it to another.

In [7]:
# ============================================================================
# CROSS-SEASON THRESHOLD TRANSFER
# ============================================================================
season_list = [s for s in season_table["season"] if s in oof["season"].unique()]
rows = []
for src in season_list:
    g_src = oof[oof["season"] == src]
    thr, _ = tk.choose_threshold(g_src[LABEL], g_src["prob"],
                                 budget_false_alert_rate=BUDGET)
    for dst in season_list:
        g = oof[oof["season"] == dst]
        m = tk.classification_metrics(g[LABEL], g["prob"], thr)
        rows.append({"threshold_from": src, "applied_to": dst,
                     "threshold": round(thr, 4), "precision": round(m["precision"], 3),
                     "recall": round(m["recall"], 3),
                     "far": round(m["false_alert_rate"], 4),
                     "alert_rate": round(m["alerts"] / max(m["n"], 1), 4)})
transfer = pd.DataFrame(rows)
print("alert rate when a threshold is moved between seasons:")
print(transfer.pivot(index="threshold_from", columns="applied_to",
                     values="alert_rate").round(4).to_string())
print("\nrecall:")
print(transfer.pivot(index="threshold_from", columns="applied_to",
                     values="recall").round(3).to_string())

alert rate when a threshold is moved between seasons:
applied_to      monsoon  post_monsoon  pre_monsoon  winter
threshold_from                                            
monsoon          0.0660        0.4445       0.2134  0.5512
post_monsoon     0.0097        0.1627       0.0597  0.2375
pre_monsoon      0.0171        0.2238       0.0897  0.3092
winter           0.0060        0.1127       0.0409  0.1814

recall:
applied_to      monsoon  post_monsoon  pre_monsoon  winter
threshold_from                                            
monsoon           0.541         0.880        0.713   0.936
post_monsoon      0.184         0.575        0.374   0.678
pre_monsoon       0.267         0.689        0.478   0.777
winter            0.128         0.442        0.293   0.571


### Result — the threshold does not transfer, and the magnitude is severe

The alert-rate matrix off the diagonal:

| Threshold from ↓ / applied to → | monsoon | post-monsoon | pre-monsoon | winter |
|---|---:|---:|---:|---:|
| **monsoon** | 0.066 | 0.445 | 0.213 | **0.551** |
| **winter** | **0.006** | 0.113 | 0.041 | 0.181 |

A threshold tuned in **monsoon and deployed in winter fires on 55% of all
station-hours** — an eight-fold increase over its intended rate, and an alert
on more than every other hour at every station. In the other direction, a
**winter threshold applied in monsoon fires on 0.6%** of hours and drops
recall to 0.13.

**This is a live defect in the notebook 08 policy**, whose thresholds were
fitted on a monsoon validation window. Deployed unchanged into November it
would flood users.

**Recommended fix — a quantile-targeted alert budget.** Instead of *"fire at
p ≥ 0.42"*, specify *"fire on the top 2% of station-hours over a trailing
30-day window"*. That holds the operational alert volume fixed and adapts
automatically as prevalence shifts, with no season detector, no threshold
table and no retuning cadence. The per-season table above shows this is
already how the model should be evaluated; it should also be how it is
operated.

## 4. Feature stability across folds

A feature whose importance swings wildly between folds is fitting the period,
not the physics.

In [8]:
# ============================================================================
# IMPORTANCE STABILITY
# ============================================================================
imps = []
ran = set(folds_df["fold"])          # a fold with too few positives was skipped
for f in [x for x in folds if x["fold"] in ran]:
    tr = df[(df["timestamp_utc"] >= f["train_start"]) & (df["timestamp_utc"] < f["train_end"])]
    if tr[LABEL].sum() < 100:
        continue
    d = lgb.Dataset(tr[FEATURES], label=tr[LABEL], free_raw_data=False)
    mdl = lgb.train(params, d, num_boost_round=200)
    gain = mdl.feature_importance("gain")
    imps.append(pd.Series(gain / max(gain.sum(), 1), index=FEATURES, name=f["fold"]))

istab = pd.concat(imps, axis=1)
istab["mean"] = istab.mean(axis=1)
istab["std"] = istab.iloc[:, :-1].std(axis=1)
istab["cv"] = istab["std"] / istab["mean"].replace(0, np.nan)
top = istab.sort_values("mean", ascending=False).head(20)
print("top 20 features by mean gain share, with cross-fold variability:")
print(top[["mean", "std", "cv"]].round(4).to_string())
print("\nmost UNSTABLE among the top 40 (high CV = period-specific):")
print(istab.sort_values("mean", ascending=False).head(40).sort_values(
    "cv", ascending=False).head(8)[["mean", "cv"]].round(4).to_string())

top 20 features by mean gain share, with cross-fold variability:
                            mean     std      cv
pm25_roll_p95_24h         0.1611  0.0166  0.1032
pm25_roll_max_24h         0.1578  0.0195  0.1236
pm25_roll_p90_24h         0.1131  0.0442  0.3911
pm25_roll_std_168h        0.0499  0.0071  0.1414
pm25_roll_std_72h         0.0282  0.0057  0.2027
pm25_roll_mean_168h       0.0241  0.0025  0.1043
pm25_roll_mean_24h        0.0215  0.0063  0.2927
elevation_m               0.0162  0.0032  0.1940
cams_no2_roll_168h        0.0152  0.0040  0.2653
cams_aod_roll_168h        0.0142  0.0036  0.2567
doy                       0.0141  0.0040  0.2834
cams_pm25_roll_168h       0.0135  0.0034  0.2510
elevation_relief_10km     0.0130  0.0036  0.2730
elevation_std_10km        0.0122  0.0030  0.2456
days_to_diwali            0.0121  0.0080  0.6592
cams_aod_roll_24h         0.0119  0.0028  0.2373
neighbour_distance_km     0.0117  0.0039  0.3317
sin_doy                   0.0103  0.0007  0.0648
pm25

### Result — stable physics, seasonal drivers, and a note on satellite

The low-CV features are the model's durable core: `pm25_roll_p95_24h`
(CV 0.10), `pm25_roll_max_24h` (0.12), `pm25_roll_std_168h` (0.14),
`sin_doy` (0.06). Recent PM2.5 distribution shape carries the model in every
period.

The high-CV features are seasonal by nature — `days_after_diwali` (CV 0.73),
`days_to_diwali` (0.66). A high CV there is **correct behaviour**, not
instability: Diwali features should matter in one window and nowhere else. A
festival feature with flat importance would be the suspicious one.

**Worth noting against notebook 05's ablation:** trained across all seasons,
CAMS satellite features *do* surface in the top 20 —
`cams_no2_roll_168h`, `cams_aod_roll_168h`, `cams_pm25_roll_168h`, each at
~1.4% gain with moderate CV — alongside `elevation_m` and
`elevation_relief_10km`. Notebook 05's monsoon-only ablation showed the
satellite rung as net-negative. Both readings are correct for their scope,
and together they say satellite and terrain are **modest, broadly stable
contributors** rather than either major levers or dead weight.

Fire transport features do **not** appear in the top 20 even across folds,
which is the strongest evidence yet that the 134-column fire block is not
earning its complexity at the current radii and decay windows.

## 5. Persist

In [9]:
# ============================================================================
# PERSIST
# ============================================================================
P9 = CFG["artifact_root"] / "phase9_rolling"
P9.mkdir(parents=True, exist_ok=True)
folds_df.to_csv(P9 / "rolling_origin_folds.csv", index=False)
season_table.to_csv(P9 / "per_season_performance.csv", index=False)
transfer.to_csv(P9 / "cross_season_threshold_transfer.csv", index=False)
istab.reset_index().rename(columns={"index": "feature"}).to_csv(
    P9 / "feature_importance_stability.csv", index=False)
oof.to_parquet(P9 / "out_of_fold_predictions.parquet", index=False)

worst = folds_df.loc[folds_df["lift"].idxmin()]
tk.write_report(P9 / "phase9_report.json", {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "label": LABEL,
    "dataset_span_days": int(span_days),
    "n_folds": len(folds_df),
    "fold_config": {"min_train_days": 180, "valid_days": 75, "embargo_hours": 48},
    "folds": folds_df.to_dict("records"),
    "prevalence_swing": {"min": float(folds_df["prevalence"].min()),
                         "max": float(folds_df["prevalence"].max()),
                         "ratio": float(folds_df["prevalence"].max()
                                        / max(folds_df["prevalence"].min(), 1e-9))},
    "stability": {c: {"mean": float(folds_df[c].mean()), "std": float(folds_df[c].std()),
                      "min": float(folds_df[c].min())}
                  for c in ("pr_auc", "lift", "recall", "roc_auc")},
    "worst_fold": {"fold": int(worst["fold"]), "seasons": worst["seasons"],
                   "lift": float(worst["lift"])},
    "per_season": season_table.to_dict("records"),
    "cross_season_threshold_transfer": transfer.to_dict("records"),
    "beats_persistence_folds": int((folds_df["lift"] > folds_df["persistence_lift"]).sum()),
    "critical_limitations": [
        "19 months of data contain ONE winter. A winter fold can be validated "
        "but never replicated — plan s.28's 3-5 year target is the fix.",
        "A fixed probability threshold does NOT transfer across seasons; the "
        "notebook 08 policy is fitted on monsoon and needs a quantile-targeted "
        "or seasonal formulation.",
        "PR-AUC is prevalence-sensitive; compare lift, not raw PR-AUC, across folds.",
    ],
})
print("wrote", P9 / "phase9_report.json")

wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/artifacts/anomaly/phase9_rolling/phase9_report.json


## Conclusion

**The model is more robust than the single-split evaluation suggested, and
the alert policy is less robust.**

**ROC-AUC varies by under 0.06 across the whole year** (0.843–0.895, std
0.019). Ranking quality is season-independent. The model beats persistence
on lift in **5 of 5 folds**, including monsoon — reversing notebook 03's
result, where the residual detector *lost* to persistence. Plan section 34's
"no major season should collapse" is satisfied on the only metric that can
fairly test it.

**Two of five folds clear the plan's joint acceptance gates outright**
(precision ≥ 0.25 and recall ≥ 0.70), and they are the higher-prevalence
folds — the periods when a public-health alert matters most. The folds that
miss are monsoon-weighted, where precision is arithmetically capped by a 3%
base rate.

**The threshold defect is severe and is the top action item.** A
monsoon-tuned threshold fires on **55% of station-hours in winter**; a
winter-tuned one fires on **0.6%** in monsoon and drops recall to 0.13. The
notebook 08 policy is monsoon-fitted and would flood users in November.
Replace the fixed probability cut with a **quantile-targeted alert budget**
(*"top N% of station-hours over a trailing 30 days"*) — self-adapting, no
season detector, no threshold table.

**The alert budget itself should be seasonal.** Winter precision at a 5%
alert rate is ~0.91 while recall is only 0.18. The budget is far too tight
for winter and roughly right for monsoon; widening it in winter costs almost
no precision.

**One winter is the binding data constraint.** No fold engineering
manufactures a second one, so fold 2's strong winter result cannot be
distinguished from a property of 2025-26's particular meteorology. Plan
section 28's 3–5 year target is right — not because more data trains a better
model, but because a single observation of the most important season cannot
be validated.

**Method note for plan section 4:** it specifies rolling-origin folds but not
how to compare them. With prevalence swinging 10×, raw PR-AUC comparisons
between folds are meaningless. Compare **ROC-AUC** for stability and **lift
over prevalence** for skill.